# Topic: SQL Unpivot Pattern

## Definition (30-second explanation)
* The Unpivot operation is the exact inverse of a Pivot; it reshapes data from a wide format (many columns) to a long/tall format (fewer columns, more rows).
* For example, converting three separate columns for `Q1_revenue`, `Q2_revenue`, and `Q3_revenue` into two columns: one for `quarter` (the label) and one for `revenue` (the value).
* The most universal way to achieve this across all SQL dialects is by stacking `SELECT` statements using `UNION ALL`.

## Why Interviewers Ask This
* Tests your ability to normalize messy, spreadsheet-style data into proper relational database formats.
* Validates your understanding of `UNION ALL` versus `UNION` and how to manipulate dataset structures for downstream analytics (like Tableau or Python, which often require long-format data).

## Core Concepts
* **The `UNION ALL` Method:** Creating a separate `SELECT` statement for every column you want to unpivot, explicitly hardcoding the category label as a string, and stacking them.
* **Category Labeling:** You must explicitly inject a string literal (e.g., `'Q1' AS quarter`) in each `SELECT` block so you don't lose track of which column the value originated from.
* **Native vs Universal:** SQL Server has a native `UNPIVOT` operator, and PostgreSQL uses `UNNEST()` with arrays, but `UNION ALL` works universally everywhere (including MySQL).

## When to Use
* Normalizing wide spreadsheet data (e.g., months as columns) before loading it into a data warehouse.
* Making data compatible with BI visualization tools (Tableau, PowerBI) that inherently prefer long-format data for axis dimensions.
* Preparing data for time-series analysis or window functions that require sequential, row-based processing.

## Advantages
* The `UNION ALL` method is universally supported across every SQL engine.
* It gives you absolute, granular control over how each column is mapped and transformed during the unpivot process.

## Limitations
* **Extremely Verbose:** If you have to unpivot 50 columns (e.g., weekly data), you have to write 50 `UNION ALL` blocks. Dynamic SQL is required for automation.
* **Performance:** Scanning the same base table multiple times (once per `UNION ALL` branch) can be computationally expensive on massive datasets.

## Common Comparisons
* **`UNION ALL` vs `UNION`:** You must use `UNION ALL` for unpivoting. `UNION` implicitly performs a deduplication step, which is computationally expensive and will incorrectly delete valid duplicate values in your data.
* **Pivot vs Unpivot:** Pivot aggregates rows into columns (using `SUM`/`MAX` + `CASE WHEN`). Unpivot breaks columns into rows (using `UNION ALL`).

## Common Interview Traps
* **Forgetting Column Alignment:** Every branch of the `UNION ALL` must have the exact same number of columns in the exact same order with compatible data types.
* **Mishandling NULLs:** A `UNION ALL` unpivot will keep `NULL` values as actual rows in the new table. You must actively decide to filter them out using a `WHERE col IS NOT NULL` clause if the business doesn't want them.

## Python / SQL Syntax
```sql
    -- Universal Unpivot using UNION ALL
    SELECT department, 'Q1' AS quarter, Q1_revenue AS revenue FROM sales_wide
    UNION ALL
    SELECT department, 'Q2' AS quarter, Q2_revenue AS revenue FROM sales_wide
    UNION ALL
    SELECT department, 'Q3' AS quarter, Q3_revenue AS revenue FROM sales_wide
    UNION ALL
    SELECT department, 'Q4' AS quarter, Q4_revenue AS revenue FROM sales_wide
    ORDER BY department, quarter;
```

## 45-Second Interview Answer
"To unpivot data from a wide format to a long format in standard SQL, I use the `UNION ALL` approach. I write a `SELECT` statement for each column I want to convert, manually hardcoding the column name as a string literal to act as the new category label, and then stack them together. I always ensure I use `UNION ALL` instead of `UNION` to preserve duplicate values and save on the performance cost of sorting. If the business requires it, I also wrap this in a CTE to filter out any resulting NULL rows."

## Example Questions:

### Q1: A student table has columns: student_id, math_score, science_score, english_score. Unpivot it into: student_id, subject, score.

* **Ideal Interview Answer:** I will use three `UNION ALL` blocks, one for each subject, hardcoding the subject name as a string literal in each block.
```sql
    SELECT student_id, 'math' AS subject, math_score AS score FROM student_grades
    UNION ALL
    SELECT student_id, 'science' AS subject, science_score AS score FROM student_grades
    UNION ALL
    SELECT student_id, 'english' AS subject, english_score AS score FROM student_grades
    ORDER BY student_id, subject;
```
* **Common Mistakes:** Using `UNION` instead of `UNION ALL`. If a student got an 85 in both math and science, `UNION` would delete one of those rows!
* **Likely Interviewer Follow-up:** What if the `science_score` column is a `FLOAT` and `math_score` is an `INT`? (Answer: In strictly typed databases, `UNION ALL` will fail if data types mismatch. I would need to explicitly `CAST(math_score AS FLOAT)` so all branches align).

### Q2: A products table has: product_id, price_2021, price_2022, price_2023. Unpivot it and calculate year-over-year price change.

* **Ideal Interview Answer:** I will unpivot the table using a CTE with `UNION ALL`. In the main query, I will use the `LAG()` window function to grab the previous year's price and calculate the difference.
```sql
    WITH UnpivotedPrices AS (
        SELECT product_id, 2021 AS price_year, price_2021 AS price FROM products
        UNION ALL
        SELECT product_id, 2022 AS price_year, price_2022 AS price FROM products
        UNION ALL
        SELECT product_id, 2023 AS price_year, price_2023 AS price FROM products
    )
    SELECT product_id, 
           price_year, 
           price,
           price - LAG(price) OVER(PARTITION BY product_id ORDER BY price_year) AS yoy_change
    FROM UnpivotedPrices;
```
* **Common Mistakes:** Storing the `price_year` as a string (e.g., `'2021'`) instead of an integer. While it sorts correctly alphabetically, storing it as an integer makes downstream math or filtering much easier.
* **Likely Interviewer Follow-up:** How would you handle a product that didn't exist in 2021 (the value is NULL)? (Answer: I would add `WHERE price IS NOT NULL` either inside each `UNION ALL` branch or in the final outer query, so the `LAG()` function correctly calculates the difference between 2022 and 2023).

### Q3: Unpivot a contact table (contact_id, phone1, phone2, email1, email2) into contact_id, contact_type, contact_value.

* **Ideal Interview Answer:** I'll use four `UNION ALL` blocks, assigning the appropriate `contact_type` label to each. Because we only want valid contact info, I'll wrap the unpivot in a CTE and filter out NULLs.
```sql
    WITH UnpivotedContacts AS (
        SELECT contact_id, 'phone' AS contact_type, phone1 AS contact_value FROM contacts
        UNION ALL
        SELECT contact_id, 'phone' AS contact_type, phone2 AS contact_value FROM contacts
        UNION ALL
        SELECT contact_id, 'email' AS contact_type, email1 AS contact_value FROM contacts
        UNION ALL
        SELECT contact_id, 'email' AS contact_type, email2 AS contact_value FROM contacts
    )
    SELECT contact_id, contact_type, contact_value
    FROM UnpivotedContacts
    WHERE contact_value IS NOT NULL;
```
* **Common Mistakes:** Trying to unpivot `phone` and `email` into two *separate* columns in the long format. True long format requires all values to go into a single `contact_value` column, utilizing the `contact_type` label to distinguish them.
* **Likely Interviewer Follow-up:** Are there performance concerns with this query? (Answer: Yes. We are doing a full table scan four times. On a massive table, this could be very slow compared to native unpivot operators).

### Q4: After unpivoting, write a window function to calculate the rank of each quarter's revenue within each department.

* **Ideal Interview Answer:** I will use the `UNION ALL` strategy in a CTE to transform the wide sales data into long format. Then, I'll use `RANK() OVER()` partitioned by the department.
```sql
    WITH LongSales AS (
        SELECT department, 'Q1' AS quarter, Q1_revenue AS revenue FROM sales_wide
        UNION ALL
        SELECT department, 'Q2' AS quarter, Q2_revenue AS revenue FROM sales_wide
        UNION ALL
        SELECT department, 'Q3' AS quarter, Q3_revenue AS revenue FROM sales_wide
        UNION ALL
        SELECT department, 'Q4' AS quarter, Q4_revenue AS revenue FROM sales_wide
    )
    SELECT department, 
           quarter, 
           revenue,
           RANK() OVER(PARTITION BY department ORDER BY revenue DESC) AS revenue_rank
    FROM LongSales
    WHERE revenue IS NOT NULL;
```
* **Common Mistakes:** Forgetting to partition by `department`, which would calculate the rank across the entire company instead of within the specific department.
* **Likely Interviewer Follow-up:** What is the difference between `RANK()` and `DENSE_RANK()` if Q1 and Q2 had the exact same revenue? (Answer: `RANK()` will assign both as rank 1, and the next quarter will be rank 3. `DENSE_RANK()` would assign both as rank 1, and the next quarter as rank 2).